# Proyecto G07 — Ingesta y validación de la capa Bronze (Danper)

**Curso:** Big Data y Analítica de Datos  
**Infraestructura:** Hadoop HDFS *on-premise*, Debian 12  
**Objetivo:** evidenciar el almacenamiento íntegro de fuentes agrocomerciales originales en la zona `/danper/bronze`, conservando su formato y procedencia.

> **Seguridad:** este Notebook solo consulta archivos y realiza una **simulación** del script. No ejecuta `--ejecutar`, no elimina archivos y no formatea HDFS. Se debe ejecutar en la máquina Debian que ya tiene los datasets y Hadoop operativos.

## 1. Arquitectura de datos

El flujo de trabajo previsto es **fuentes originales (CSV, XLS, XLSX, PDF) → ingesta → HDFS Bronze → procesamiento Silver → información Gold**.

- **Bronze:** conserva los archivos originales sin limpieza ni transformación analítica.
- **Silver:** incorpora depuración, normalización, tipificación y estructuración para el análisis; es trabajo independiente del equipo.
- **Gold:** agregaciones e indicadores útiles para decisiones agrocomerciales.

La solución usa infraestructura local (*on-premise*), no AWS. La replicación actual de HDFS es 1, por lo que los datos deben conservarse también fuera de HDFS como respaldo.

## 2. Preparación y comprobación del entorno

Se detecta la raíz del repositorio desde el Notebook, independientemente de si Jupyter inicia en el directorio del proyecto o en `notebooks/`.

In [1]:
from pathlib import Path
import subprocess
import hashlib
import csv
from collections import Counter, defaultdict

ubicacion = Path.cwd().resolve()
candidatos = [ubicacion, *ubicacion.parents]
REPO = next((p for p in candidatos if (p / "scripts" / "ingesta_bronze.py").is_file()), None)
if REPO is None:
    raise FileNotFoundError("Abra Jupyter desde la carpeta del repositorio G07 o desde su subcarpeta notebooks")

DATASETS = Path.home() / "proyecto_danper" / "datasets"
BRONZE = "/danper/bronze"
print("Repositorio:", REPO)
print("Datos locales:", DATASETS)
print("Zona HDFS:", BRONZE)
print("Origen local disponible:", DATASETS.is_dir())

Repositorio: /home/fao/G07-BigData-AnaliticaAgrocomercial-Danper
Datos locales: /home/fao/proyecto_danper/datasets
Zona HDFS: /danper/bronze
Origen local disponible: True


In [2]:
def hdfs(*args, verificar=True):
    resultado = subprocess.run(
        ["hdfs", *args], text=True, capture_output=True,
        timeout=180
    )
    if verificar and resultado.returncode != 0:
        raise RuntimeError(resultado.stderr.strip() or resultado.stdout.strip())
    return resultado

print("HDFS accesible:", hdfs("dfs", "-test", "-d", BRONZE, verificar=False).returncode == 0)
print("Replicacion configurada:", hdfs("getconf", "-confKey", "dfs.replication").stdout.strip())
print(hdfs("dfsadmin", "-safemode", "get").stdout.strip())

HDFS accesible: True
Replicacion configurada: 1
Safe mode is OFF


## 3. Inventario de fuentes locales

Se cuentan los **archivos por ruta**, no los contenidos únicos. En la ENA 2025 existen cuestionarios PDF repetidos en distintos módulos; se conservan sus rutas para mantener trazabilidad. Las cifras se calculan en tiempo de ejecución.

In [3]:
if not DATASETS.is_dir():
    raise FileNotFoundError(f"No existe la carpeta local: {DATASETS}")
archivos_locales = sorted(p for p in DATASETS.rglob("*") if p.is_file())
por_categoria = Counter(p.relative_to(DATASETS).parts[0] for p in archivos_locales)
por_formato = Counter(p.suffix.lower() or "sin_extension" for p in archivos_locales)
print("TOTAL DE ARCHIVOS LOCALES:", len(archivos_locales))
print("\nDistribucion por categoria:")
for categoria, total in sorted(por_categoria.items()):
    print(f"  {categoria:20s} {total:3d}")
print("\nDistribucion por formato:")
for formato, total in sorted(por_formato.items()):
    print(f"  {formato:12s} {total:3d}")

TOTAL DE ARCHIVOS LOCALES: 180

Distribucion por categoria:
  abastecimiento        57
  agroindustria          2
  costos                94
  exportaciones         11
  pecuaria               1
  produccion            12
  sostenibilidad         3

Distribucion por formato:
  .csv          36
  .pdf         124
  .xls           1
  .xlsx         19


## 4. Inventario de archivos en HDFS Bronze

La consulta recursiva se hace **una sola vez**, evitando iniciar procesos Java por cada archivo. Se recopilan las rutas y tamaños de los archivos, sin descargarlos.

In [4]:
salida = hdfs("dfs", "-ls", "-R", BRONZE).stdout
archivos_hdfs = {}
for linea in salida.splitlines():
    partes = linea.split(maxsplit=7)
    if len(partes) == 8 and partes[0].startswith("-"):
        archivos_hdfs[partes[7]] = int(partes[4])
print("TOTAL DE ARCHIVOS EN HDFS BRONZE:", len(archivos_hdfs))
print("TAMANIO TOTAL:", round(sum(archivos_hdfs.values()) / 1024**2, 2), "MiB")
print("Rutas de muestra:")
for ruta in sorted(archivos_hdfs)[:5]:
    print(" ", ruta)

TOTAL DE ARCHIVOS EN HDFS BRONZE: 180
TAMANIO TOTAL: 576.41 MiB
Rutas de muestra:
  /danper/bronze/abastecimiento/6342029-reporte-de-ingreso-y-precios-en-el-gran-mercado-mayorista-de-lima-31-01-2025.pdf
  /danper/bronze/abastecimiento/7456609-reporte-de-ingreso-y-precios-en-el-gran-mercado-mayorista-de-lima-31-12-2025.pdf
  /danper/bronze/abastecimiento/gmml_2024/4987850-reporte-de-ingreso-y-precios-en-el-gran-mercado-mayorista-de-lima-31-01-2024.pdf
  /danper/bronze/abastecimiento/gmml_2024/5139659-reporte-de-ingreso-y-precios-en-el-gran-mercado-mayorista-de-lima-29-02-2024.pdf
  /danper/bronze/abastecimiento/gmml_2024/5282101-reporte-de-ingreso-y-precios-en-el-gran-mercado-mayorista-de-lima-28-03-2024.pdf


## 5. Distribución por categoría analítica

Se agrupan las rutas lógicas de HDFS de acuerdo con la procedencia de los datasets. Los reportes regionales MINCETUR se almacenan bajo `comercio_regional/mincetur`, pero en el inventario analítico cuentan como exportaciones.

In [5]:
def categoria_hdfs(ruta):
    ruta_relativa = ruta.removeprefix(BRONZE + "/")
    principal = ruta_relativa.split("/", 1)[0]
    return "exportaciones" if principal == "comercio_regional" else principal

conteo_hdfs = Counter(categoria_hdfs(ruta) for ruta in archivos_hdfs)
print(f"{'CATEGORIA':22s} {'LOCAL':>8s} {'HDFS':>8s}")
print("-" * 40)
for categoria in sorted(set(por_categoria) | set(conteo_hdfs)):
    print(f"{categoria:22s} {por_categoria[categoria]:8d} {conteo_hdfs[categoria]:8d}")
print("-" * 40)
print(f"{'TOTAL':22s} {len(archivos_locales):8d} {len(archivos_hdfs):8d}")
assert len(archivos_locales) == len(archivos_hdfs), "Hay diferencias entre cantidad local y HDFS"
assert all(por_categoria[c] == conteo_hdfs[c] for c in por_categoria), "Difieren cantidades por categoria"
print("Conteos por categoria coherentes.")

CATEGORIA                 LOCAL     HDFS
----------------------------------------
abastecimiento               57       57
agroindustria                 2        2
costos                       94       94
exportaciones                11       11
pecuaria                      1        1
produccion                   12       12
sostenibilidad                3        3
----------------------------------------
TOTAL                       180      180
Conteos por categoria coherentes.


## 6. Verificación de idempotencia: simulación segura

El script `scripts/ingesta_bronze.py` usa el modo simulación **sin** `--ejecutar`: detecta qué rutas ya existen y cuáles estarían pendientes. La simulación no modifica HDFS. Una ejecución con 180 existentes y cero pendientes demuestra que la carga es completa según el inventario de rutas y tamaños; no vuelve a calcular los 180 SHA-256 en este paso.

In [6]:
import sys
resultado = subprocess.run(
    [sys.executable, str(REPO / "scripts" / "ingesta_bronze.py")],
    cwd=str(REPO), text=True, capture_output=True, timeout=240
)
print(resultado.stdout)
if resultado.returncode != 0:
    print(resultado.stderr)
    raise RuntimeError("La simulacion de Bronze devolvio un error")
assert "Pendientes: 0" in resultado.stdout, "La simulacion detecto archivos pendientes"
assert "Ya existentes: 180" in resultado.stdout, "Revisar conteo de archivos existentes"
print("Simulacion finalizada sin escritura.")

===== BRONZE DANPER =====
Modo: SIMULACION - SIN ESCRITURA
Archivos locales: 180
Consultando inventario de HDFS...
Archivos encontrados en HDFS: 180

===== RESUMEN POR CATEGORIA =====
abastecimiento: 57
agroindustria: 2
costos: 94
exportaciones: 11
pecuaria: 1
produccion: 12
sostenibilidad: 3

===== REVISION =====
Ya existentes: 180
Pendientes: 0
Total: 180

Simulacion terminada.
No se modifico HDFS.
Para realizar la ingesta: python3 scripts/ingesta_bronze.py --ejecutar

Simulacion finalizada sin escritura.


## 7. Integridad de bloques mediante HDFS FSCK

`fsck` comprueba la disponibilidad de los bloques dentro de HDFS. **HEALTHY no sustituye la comparación SHA-256 del script**, pero confirma que los archivos tienen bloques accesibles según HDFS.

In [7]:
resultado_fsck = hdfs("fsck", BRONZE, "-files", verificar=False)
texto_fsck = resultado_fsck.stdout + "\n" + resultado_fsck.stderr
print("\n".join(texto_fsck.splitlines()[-18:]))
assert resultado_fsck.returncode == 0 and "is HEALTHY" in texto_fsck, (
    "HDFS no esta HEALTHY; revisar fsck antes de continuar"
)
print("Estado de bloques: HEALTHY")

 Total size:	0 B
 Total files:	0
 Total block groups (validated):	0
 Minimally erasure-coded block groups:	0
 Over-erasure-coded block groups:	0
 Under-erasure-coded block groups:	0
 Unsatisfactory placement block groups:	0
 Average block group size:	0.0
 Missing block groups:		0
 Corrupt block groups:		0
 Missing internal blocks:	0
 Blocks queued for replication:	0
FSCK ended at Sun Oct 11 00:55:44 COT 2026 in 69 milliseconds


The filesystem under path '/danper/bronze' is HEALTHY

Connecting to namenode via http://localhost:9870/fsck?ugi=fao&files=1&path=%2Fdanper%2Fbronze
Estado de bloques: HEALTHY


## 8. Trazabilidad y duplicados por contenido

Se realiza el cálculo SHA-256 **local**, sin cambiar los datasets. Un mismo contenido puede repetirse en diversas carpetas de módulos ENA; eso no implica necesariamente un error de ingesta. Para datasets repetidos, la ubicación original constituye parte de su procedencia.

In [8]:
def calcular_sha256(ruta):
    digest = hashlib.sha256()
    with ruta.open("rb") as entrada:
        for bloque in iter(lambda: entrada.read(1024 * 1024), b""):
            digest.update(bloque)
    return digest.hexdigest()

hashes = defaultdict(list)
for archivo in archivos_locales:
    hashes[calcular_sha256(archivo)].append(archivo.relative_to(DATASETS).as_posix())
grupos_repetidos = {sha: rutas for sha, rutas in hashes.items() if len(rutas) > 1}
print("Archivos por ruta:", len(archivos_locales))
print("Contenidos SHA-256 distintos:", len(hashes))
print("Grupos de contenido repetido:", len(grupos_repetidos))
for numero, rutas in enumerate(grupos_repetidos.values(), 1):
    print(f"Grupo {numero}: {len(rutas)} rutas; ejemplo: {rutas[0]}")

Archivos por ruta: 180
Contenidos SHA-256 distintos: 142
Grupos de contenido repetido: 2
Grupo 1: 19 rutas; ejemplo: costos/ena_2025/ENA_2025/1035-Modulo1893/01_CUESTIONARIO_ENA_2025.pdf
Grupo 2: 21 rutas; ejemplo: costos/ena_2025/ENA_2025/1035-Modulo1893/02_CUESTIONARIO_EMPRESA_2025.pdf


## 9. Validación adicional opcional: comparación de contenidos

La ejecución real de `ingesta_bronze.py --ejecutar` ya verifica SHA-256 de cada archivo nuevo después de subirlo y de los preexistentes antes de cargar. Esta celda **solo lee** una muestra de HDFS y compara su SHA-256 con los originales locales. Se eligen las seis provincias iniciales, de tamaño pequeño, para que la comprobación sea rápida.

In [9]:
provincias = ["ASCOPE", "CHEPEN", "GRAN_CHIMU", "PACASMAYO", "TRUJILLO", "VIRU"]
for provincia in provincias:
    nombre = f"PROVINCIA {provincia}_2024-2025.xlsx"
    local = DATASETS / "produccion" / nombre
    remoto = f"{BRONZE}/produccion/{nombre}"
    contenido = subprocess.run(
        ["hdfs", "dfs", "-cat", remoto], capture_output=True, timeout=90
    )
    if contenido.returncode != 0:
        raise RuntimeError(f"No se puede leer {remoto}: {contenido.stderr.decode(errors='replace')}")
    sha_local = calcular_sha256(local)
    sha_remoto = hashlib.sha256(contenido.stdout).hexdigest()
    print(f"{provincia:20s}", "COINCIDE" if sha_local == sha_remoto else "DIFERENTE")
    assert sha_local == sha_remoto, f"Fallo de integridad: {remoto}"
print("Seis archivos comparados correctamente.")

ASCOPE               COINCIDE
CHEPEN               COINCIDE
GRAN_CHIMU           COINCIDE
PACASMAYO            COINCIDE
TRUJILLO             COINCIDE
VIRU                 COINCIDE
Seis archivos comparados correctamente.


## 10. Conclusiones técnicas

1. La ingesta conservó las fuentes sin transformaciones analíticas, respetando el propósito de **Bronze**.
2. La separación por categorías facilita la trazabilidad y la posterior extracción hacia **Silver**.
3. La validación de rutas y tamaños evita sobrescrituras accidentales; el cotejo **SHA-256** comprueba igualdad de contenido durante la carga y en la muestra posterior.
4. La **idempotencia** permite repetir la simulación sin insertar nuevamente archivos existentes.
5. `fsck` confirma el estado de los bloques en HDFS. Con replicación **1**, es indispensable conservar un respaldo local y revisar la persistencia del DataNode antes de depender de HDFS como único almacenamiento.
6. Las fuentes Bronze están disponibles en esta máquina Debian; **Git no distribuye los datasets ni sincroniza los bloques de HDFS** con las máquinas de otros integrantes.

**Evidencia de ejecución previa:** 180 rutas cargadas (6 verificadas y 174 nuevas), 180 en HDFS, `fsck` **HEALTHY**. Estos resultados proceden del registro de terminal de la ingesta realizada, y las celdas del Notebook permiten verificarlos nuevamente.